# Dendros Quickstart

This notebook demonstrates the main workflows of the **`dendros`** library
for reading and exploring [Galacticus](https://github.com/galacticusorg/galacticus)
HDF5 output files.

We begin by creating a small synthetic HDF5 file that mimics the Galacticus
output structure, so you can run this notebook without needing a real
Galacticus run.

In [ ]:
import tempfile
from pathlib import Path

import h5py
import numpy as np

# -----------------------------------------------------------------------
# Create a minimal synthetic Galacticus-like HDF5 file
# -----------------------------------------------------------------------

tmpdir = tempfile.mkdtemp()
fpath = str(Path(tmpdir) / "galacticus.hdf5")

rng = np.random.default_rng(42)
N1, N2, N3 = 500, 300, 150  # galaxy counts for three snapshots

# Galaxy IDs (nodeUniqueIDBranchTip) overlap between snapshots so histories
# can be traced.  Output1 is the latest (z=0).
ids_1 = np.arange(N1, dtype=np.int64)
ids_2 = np.arange(N2, dtype=np.int64)           # first N2 survivors from O1
ids_3 = np.arange(N3, dtype=np.int64)           # first N3 survivors from O2

with h5py.File(fpath, "w") as f:
    f.attrs["statusCompletion"] = 0

    outputs = f.create_group("Outputs")

    def _write(grp_name, t, a, ids, N):
        grp = outputs.create_group(grp_name)
        grp.attrs["outputTime"] = t
        grp.attrs["outputExpansionFactor"] = a
        nd = grp.create_group("nodeData")
        nd.create_dataset("nodeUniqueIDBranchTip", data=ids)
        ds = nd.create_dataset("basicMass", data=rng.lognormal(26.0, 1.0, N))
        ds.attrs["comment"] = "Virial mass of the halo"
        ds.attrs["unitsInSI"] = 1.989e30
        ds = nd.create_dataset("diskMassStellar", data=rng.lognormal(23.0, 1.0, N))
        ds.attrs["comment"] = "Disk stellar mass"
        ds.attrs["unitsInSI"] = 1.989e30
        ds = nd.create_dataset("diskRadius", data=rng.lognormal(-3.0, 0.5, N))
        ds.attrs["comment"] = "Half-mass radius of the disk"
        ds.attrs["unitsInSI"] = 3.086e22

    # Output1 (z=0), Output2 (z=1), Output3 (z~3) — ordered by increasing
    # index but decreasing scale factor, matching Galacticus convention.
    _write("Output1", 13.8, 1.0, ids_1, N1)
    _write("Output2",  5.9, 0.5, ids_2, N2)
    _write("Output3",  2.1, 0.25, ids_3, N3)

print(f"Synthetic file written to: {fpath}")


## Opening a file

In [ ]:
import sys
import os
try:
    notebook_path = __session__
except NameError:
    # Fallback to environment variable
    notebook_path = os.environ.get("JPY_SESSION_NAME")
module_path = os.path.abspath(os.path.dirname(notebook_path) + '/../src')
if module_path not in sys.path:
    sys.path.insert(0,module_path)

In [ ]:
from dendros import open_outputs
c = open_outputs(fpath)
print(c)

## Checking completion status

In [ ]:
c.validate_completion()  # raises RuntimeError if the run did not complete
print("Run completed successfully.")

## Listing outputs

`list_outputs()` returns an `astropy.table.Table` (default), a
`pandas.DataFrame` or a `tabulate` string with time, scale factor, and redshift for each snapshot.

In [ ]:
tbl = c.list_outputs()
print(tbl)

In [ ]:
# Access individual output metadata
for meta in c.outputs:
    print(f"{meta.name}: z = {meta.redshift:.3f}, t = {meta.time:.2f} Gyr")

## Listing properties

`list_properties()` shows the datasets available in the `nodeData` group,
together with their description and SI conversion factor.

In [ ]:
props = c.list_properties("Output1",format="tabulate")
print(props)

## Reading datasets

In [ ]:
# Read by dataset path — same string used as dict key
data = c.read("Output1", ["nodeData/basicMass", "nodeData/diskMassStellar"])
print("basicMass shape:", data["nodeData/basicMass"].shape)

# Or use a dict for custom labels
data = c.read(
    "Output1",
    {"Mhalo": "nodeData/basicMass", "Mstar": "nodeData/diskMassStellar"},
)
print("Mhalo[:5] =", data["Mhalo"][:5])

## Filtering galaxies

Pass a boolean mask or integer index array as `where`.

In [ ]:
all_data = c.read("Output1", {"Mhalo": "nodeData/basicMass"})
massive = all_data["Mhalo"] > 1.0e12

filtered = c.read(
    "Output1",
    {"Mhalo": "nodeData/basicMass", "Mstar": "nodeData/diskMassStellar"},
    where=massive,
)
print(f"Selected {massive.sum()} of {len(all_data['Mhalo'])} galaxies")

## h5py-like browsing

In [ ]:
print("Top-level keys:", c.keys())

grp = c["Outputs/Output1"]
print("Output1 attrs:", grp.attrs)
print("Output1 keys:", grp.keys())

ds = c["Outputs/Output1/nodeData/basicMass"]
print(f"haloMass dtype={ds.dtype}  shape={ds.shape}")

## Tracing galaxy histories

`nodeUniqueIDBranchTip` uniquely identifies a galaxy within a file and is
constant across outputs, so `trace_history` can assemble per-galaxy time
series from all snapshots in a single call. The returned arrays have an
extra trailing axis indexing time; slots where the galaxy was absent are
filled with `NaN` (floats), `int_sentinel` (ints), or `False` (bools), and
a companion `present` mask is returned as the canonical indicator of
presence.

In [ ]:
ids = [0, 150, 300]   # a galaxy that survives to z=0, one that merges before z=1,
                      # and one that only exists in the earliest snapshot
hist = c.trace_history(
    ids,
    {"Mhalo": "nodeData/basicMass", "Mstar": "nodeData/diskMassStellar"},
)

print("outputs:", hist["output_names"].tolist())
print("present mask:\n", hist["present"])
print("Mstar shape:", hist["Mstar"].shape)
print("time shape :", hist["time"].shape)

Use the `present` mask to drop absent slots before plotting or arithmetic:

In [ ]:
for i, gid in enumerate(ids):
    mask = hist["present"][i]
    t     = hist["time"][i][mask]
    mstar = hist["Mstar"][i][mask]
    print(f"id={gid}: present at {mask.sum()} / {mask.size} outputs; Mstar = {mstar}")

## Multi-file (MPI) example

Create two synthetic MPI-split files and open them together.

In [ ]:
mpi0 = str(Path(tmpdir) / "galacticus_MPI:0000.hdf5")
mpi1 = str(Path(tmpdir) / "galacticus_MPI:0001.hdf5")

for rank, path in enumerate([mpi0, mpi1]):
    with h5py.File(path, "w") as f:
        f.attrs["statusCompletion"] = 0
        grp = f.create_group("Outputs/Output1")
        grp.attrs["outputTime"] = 13.8
        grp.attrs["outputExpansionFactor"] = 1.0
        nd = grp.create_group("nodeData")
        start = rank * 250
        ds = nd.create_dataset("basicMass", data=rng.lognormal(26.0, 1.0, 250))
        ds.attrs["comment"] = "Virial mass of the halo"
        ds.attrs["unitsInSI"] = 1.989e30

# open_outputs auto-detects the peer file from any single rank
with open_outputs(mpi0) as mc:
    print("Files in collection:", mc.files)
    data = mc.read("Output1", ["nodeData/basicMass"])
    print("Total galaxies from both ranks:", len(data["nodeData/basicMass"]))

## Clean up

In [ ]:
c.close()
import shutil
shutil.rmtree(tmpdir)
print("Done.")